Le prétraitement des données

In [1]:
import pandas as pd

df = pd.read_csv("../data/cache_filtre.csv")
print("Avant nettoyage :", df.shape)

mask_corrompu = df['label'].astype(str).str.lower() == 'label'
print("Lignes corrompues trouvées :", mask_corrompu.sum())

df_propre = df[~mask_corrompu].reset_index(drop=True)
df_propre.to_csv("../data/cache_filtre.csv", index=False)
print("Après nettoyage :", df_propre.shape)

C:\Users\user\AppData\Local\Temp\ipykernel_8588\1742350007.py:3: DtypeWarning: Columns (0: flow_duration, 1: Header_Length, 2: Protocol Type, 3: Duration, 4: Rate, 5: Srate, 6: Drate, 7: fin_flag_number, 8: syn_flag_number, 9: rst_flag_number, 10: psh_flag_number, 11: ack_flag_number, 12: ece_flag_number, 13: cwr_flag_number, 14: ack_count, 15: syn_count, 16: fin_count, 17: urg_count, 18: rst_count, 19: HTTP, 20: HTTPS, 21: DNS, 22: Telnet, 23: SMTP, 24: SSH, 25: IRC, 26: TCP, 27: UDP, 28: DHCP, 29: ARP, 30: ICMP, 31: IPv, 32: LLC, 33: Tot sum, 34: Min, 35: Max, 36: AVG, 37: Std, 38: Tot size, 39: IAT, 40: Number, 41: Magnitue, 42: Radius, 43: Covariance, 44: Variance, 45: Weight) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/cache_filtre.csv")


Avant nettoyage : (2536392, 47)
Lignes corrompues trouvées : 2
Après nettoyage : (2536390, 47)


In [2]:
import pandas as pd
import numpy as np
import os
import glob
from collections import defaultdict


FAMILLES_CIBLES = ['DDoS', 'DoS', 'Mirai', 'Recon', 'Spoofing', 'Injection']
LABEL_BENIN = 'BenignTraffic'
MAPPING_EXPLICITE = {
    'DNS_Spoofing': 'Spoofing', 'MITM-ArpSpoofing': 'Spoofing',
    'SqlInjection': 'Injection', 'CommandInjection': 'Injection',
    'XSS': 'Injection', 'Uploading_Attack': 'Injection'
}

def mapper_famille(label):
    if label == LABEL_BENIN: return LABEL_BENIN
    if label in MAPPING_EXPLICITE: return MAPPING_EXPLICITE[label]
    for famille in FAMILLES_CIBLES:
        if label.startswith(famille): return famille
    return None


DATASET_PATH = "../data/CICIoT2023"
CACHE_PATH = "../data/cache_filtre.csv"
fichiers = sorted(glob.glob(os.path.join(DATASET_PATH, "*.csv")))
compteur = defaultdict(int)
max_par_famille = 150000

# Repartir d'un fichier propre à chaque exécution du notebook
if os.path.exists(CACHE_PATH):
    os.remove(CACHE_PATH)

premiere_ecriture = True

for i, f in enumerate(fichiers, 1):
    for chunk in pd.read_csv(f, chunksize=50000):
        chunk['famille'] = chunk['label'].apply(mapper_famille)
        chunk = chunk[chunk['famille'].notna()]

        chunks_filtres = []
        for fam in chunk['famille'].unique():
            reste = max_par_famille - compteur[fam]
            if reste > 0:
                sous_chunk = chunk[chunk['famille'] == fam].head(reste)
                compteur[fam] += len(sous_chunk)
                chunks_filtres.append(sous_chunk)

        if chunks_filtres:
            chunk_final = pd.concat(chunks_filtres).drop(columns=['famille'])
            chunk_final.to_csv(CACHE_PATH, mode='a', header=premiere_ecriture, index=False)
            premiere_ecriture = False
    print(f"Processed file {i}/{len(fichiers)}")

Processed file 1/80
Processed file 2/80
Processed file 3/80
Processed file 4/80
Processed file 5/80
Processed file 6/80
Processed file 7/80
Processed file 8/80
Processed file 9/80
Processed file 10/80
Processed file 11/80
Processed file 12/80
Processed file 13/80
Processed file 14/80
Processed file 15/80
Processed file 16/80
Processed file 17/80
Processed file 18/80
Processed file 19/80
Processed file 20/80
Processed file 21/80
Processed file 22/80
Processed file 23/80
Processed file 24/80
Processed file 25/80
Processed file 26/80
Processed file 27/80
Processed file 28/80
Processed file 29/80
Processed file 30/80
Processed file 31/80
Processed file 32/80
Processed file 33/80
Processed file 34/80
Processed file 35/80
Processed file 36/80
Processed file 37/80
Processed file 38/80
Processed file 39/80
Processed file 40/80
Processed file 41/80
Processed file 42/80
Processed file 43/80
Processed file 44/80
Processed file 45/80
Processed file 46/80
Processed file 47/80
Processed file 48/80
P